# 25.07 - CV consolidation

**Notebook type:** Solution notebook with theory, complete implementations, smoke checks, and test cases.

**Daily output:** Full CV recode attempt.

**Priority:** P0 — rebuild the essential image-classification pipeline from memory.

Today is a closed-book consolidation day. Recreate the smallest complete CV workflow: inspect tensors, split without leakage, normalize from training data, build a CNN, train it, and evaluate accuracy plus Macro-F1.

## Core Ideas

- **Track the tensor contract:** images use `[N, C, H, W]`, `float32`, and labels use `[N]`, `long`.
- **Split before fitting preprocessing:** validation information must not influence training normalization.
- **Keep class indices consistent:** the final layer width, label values, and metric class order must agree.
- **Use the standard training state:** `model.train()` with gradients for optimization; `model.eval()` and `torch.no_grad()` for inference.
- **Measure more than accuracy:** Macro-F1 and per-class recall expose weak classes that overall accuracy can hide.
- **Make the pipeline reproducible:** fix seeds and hold the split constant when comparing experiments.

## Setup and Prepared Image Data

The cell below provides a deterministic three-class image dataset. The learner does not need to create fixtures. Each class has a simple bright pattern plus noise: vertical, horizontal, or diagonal.

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 25
torch.manual_seed(SEED)
np.random.seed(SEED)

IMAGE_SIZE = 16
SAMPLES_PER_CLASS = 36
NUM_CLASSES = 3
CLASS_NAMES = ["vertical", "horizontal", "diagonal"]

images = torch.zeros(NUM_CLASSES * SAMPLES_PER_CLASS, 1, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)
labels = torch.arange(NUM_CLASSES, dtype=torch.long).repeat_interleave(SAMPLES_PER_CLASS)

for row_index, label in enumerate(labels.tolist()):
    image = torch.rand(1, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32) * 0.12
    offset = (row_index % 3) - 1
    if label == 0:
        center = IMAGE_SIZE // 2 + offset
        image[:, :, center - 1:center + 2] += 0.85
    elif label == 1:
        center = IMAGE_SIZE // 2 + offset
        image[:, center - 1:center + 2, :] += 0.85
    else:
        for pixel in range(IMAGE_SIZE):
            shifted = min(max(pixel + offset, 0), IMAGE_SIZE - 1)
            image[:, pixel, shifted] += 0.85
    images[row_index] = image.clamp(0.0, 1.0)

print("images:", images.shape, images.dtype, images.device)
print("labels:", labels.shape, labels.dtype, torch.bincount(labels).tolist())

## Exercise 25-A: Split and normalize without leakage

Create a deterministic class-balanced train/validation split. Calculate normalization statistics from training images only, then apply them to both partitions.

**Return structure — `split_and_normalize(images, labels, val_fraction, seed)`:**

- Returns a `dict` with exactly six keys.
- `"train_images"`: CPU `torch.Tensor`, shape `[N_train, C, H, W]`, dtype `torch.float32`, normalized with training statistics.
- `"train_labels"`: CPU `torch.Tensor`, shape `[N_train]`, dtype `torch.long`.
- `"val_images"`: CPU `torch.Tensor`, shape `[N_val, C, H, W]`, dtype `torch.float32`, normalized with the same training statistics.
- `"val_labels"`: CPU `torch.Tensor`, shape `[N_val]`, dtype `torch.long`.
- `"mean"` and `"std"`: scalar CPU `torch.Tensor` objects; `std` is strictly positive.
- Every source row appears in exactly one partition, and every class appears in both partitions.

In [ ]:
def split_and_normalize(images, labels, val_fraction=0.25, seed=25):
    if images.ndim != 4 or labels.ndim != 1 or len(images) != len(labels):
        raise ValueError("Expected images [N,C,H,W] and matching labels [N]")
    if images.dtype != torch.float32 or labels.dtype != torch.long:
        raise TypeError("Expected float32 images and long labels")
    if images.device.type != "cpu" or labels.device.type != "cpu":
        raise ValueError("This compact exercise expects CPU tensors")
    if not 0.0 < val_fraction < 1.0:
        raise ValueError("val_fraction must be between zero and one")

    generator = torch.Generator().manual_seed(seed)
    train_parts = []
    val_parts = []
    for class_index in torch.unique(labels, sorted=True).tolist():
        class_rows = torch.where(labels == class_index)[0]
        if len(class_rows) < 2:
            raise ValueError("Every class needs at least two samples")
        class_rows = class_rows[torch.randperm(len(class_rows), generator=generator)]
        val_count = min(max(int(round(len(class_rows) * val_fraction)), 1), len(class_rows) - 1)
        val_parts.append(class_rows[:val_count])
        train_parts.append(class_rows[val_count:])

    train_indices = torch.cat(train_parts)
    val_indices = torch.cat(val_parts)
    train_indices = train_indices[torch.randperm(len(train_indices), generator=generator)]
    val_indices = val_indices[torch.randperm(len(val_indices), generator=generator)]

    raw_train = images[train_indices].clone()
    raw_val = images[val_indices].clone()
    mean = raw_train.mean()
    std = raw_train.std().clamp_min(1e-6)

    return {
        "train_images": (raw_train - mean) / std,
        "train_labels": labels[train_indices].clone(),
        "val_images": (raw_val - mean) / std,
        "val_labels": labels[val_indices].clone(),
        "mean": mean,
        "std": std,
    }


# Smoke check
smoke_split = split_and_normalize(images, labels, val_fraction=0.25, seed=SEED)
print("train:", smoke_split["train_images"].shape, smoke_split["train_labels"].shape)
print("validation:", smoke_split["val_images"].shape, smoke_split["val_labels"].shape)
print("train mean/std:", smoke_split["train_images"].mean().item(), smoke_split["train_images"].std().item())

train_loader = DataLoader(
    TensorDataset(smoke_split["train_images"], smoke_split["train_labels"]),
    batch_size=18,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    TensorDataset(smoke_split["val_images"], smoke_split["val_labels"]),
    batch_size=18,
    shuffle=False,
)

## Exercise 25-B: Rebuild a minimal CNN

Use two convolution blocks, spatial downsampling, adaptive pooling, and a linear classifier. Do not apply softmax before `CrossEntropyLoss`.

**Return structure — `MinimalCNN(num_classes)` and its call `model(batch)`:**

- Construction returns an `nn.Module` instance whose parameters are `torch.float32` by default.
- Calling it with a `torch.float32` tensor of shape `[B, 1, H, W]` returns logits as a `torch.Tensor` of shape `[B, num_classes]`.
- Output dtype and device match the model parameters; logits are raw, unnormalized scores.

In [ ]:
class MinimalCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(16, num_classes)

    def forward(self, batch):
        features = self.features(batch)
        return self.classifier(features.flatten(1))


# Smoke check
smoke_model = MinimalCNN(NUM_CLASSES)
smoke_logits = smoke_model(smoke_split["train_images"][:4])
print("logits:", smoke_logits.shape, smoke_logits.dtype, smoke_logits.device)

## Exercise 25-C: Write one training epoch

Move batches to the model device, clear gradients, compute loss, backpropagate, and update parameters. Average loss by sample count rather than by number of batches.

**Return structure — `train_one_epoch(model, loader, optimizer, criterion)`:**

- Returns one Python `float`: the non-negative mean training loss per sample across the loader.
- Side effects: puts `model` in training mode, computes gradients, and updates model parameters through `optimizer`.
- The returned value is finite when the inputs and model outputs are finite.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    device = next(model.parameters()).device
    total_loss = 0.0
    total_samples = 0

    for batch_images, batch_labels in loader:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)
        optimizer.zero_grad()
        logits = model(batch_images)
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_labels)
        total_samples += len(batch_labels)

    if total_samples == 0:
        raise ValueError("loader must contain at least one sample")
    return float(total_loss / total_samples)


# Smoke check
smoke_optimizer = torch.optim.Adam(smoke_model.parameters(), lr=0.02)
smoke_loss = train_one_epoch(smoke_model, train_loader, smoke_optimizer, nn.CrossEntropyLoss())
print("one-epoch loss:", smoke_loss)

## Exercise 25-D: Evaluate accuracy, Macro-F1, and recall

Run batched inference without gradients. Build the confusion matrix with rows as true classes and columns as predicted classes, then derive per-class recall and Macro-F1.

**Return structure — `evaluate_model(model, loader, num_classes)`:**

- Returns a `dict` with exactly seven keys.
- `"logits"`: CPU `torch.Tensor`, shape `[N, num_classes]`, dtype `torch.float32`.
- `"labels"` and `"predictions"`: CPU `torch.Tensor` objects, shape `[N]`, dtype `torch.long`.
- `"confusion_matrix"`: CPU `torch.Tensor`, shape `[num_classes, num_classes]`, dtype `torch.long`; rows are truth and columns are predictions.
- `"per_class_recall"`: `list[float]` of length `num_classes` in class-index order.
- `"accuracy"` and `"macro_f1"`: Python `float` values in `[0.0, 1.0]`.

In [ ]:
def evaluate_model(model, loader, num_classes):
    model.eval()
    device = next(model.parameters()).device
    logits_parts = []
    label_parts = []

    with torch.no_grad():
        for batch_images, batch_labels in loader:
            logits_parts.append(model(batch_images.to(device)).cpu())
            label_parts.append(batch_labels.cpu())

    if not logits_parts:
        raise ValueError("loader must contain at least one sample")
    logits = torch.cat(logits_parts)
    true_labels = torch.cat(label_parts).long()
    predictions = logits.argmax(dim=1).long()

    confusion = torch.zeros(num_classes, num_classes, dtype=torch.long)
    for truth, prediction in zip(true_labels.tolist(), predictions.tolist()):
        confusion[truth, prediction] += 1

    recalls = []
    f1_scores = []
    for class_index in range(num_classes):
        true_positive = confusion[class_index, class_index].item()
        false_negative = confusion[class_index, :].sum().item() - true_positive
        false_positive = confusion[:, class_index].sum().item() - true_positive
        recall = true_positive / max(true_positive + false_negative, 1)
        precision = true_positive / max(true_positive + false_positive, 1)
        f1 = 0.0 if precision + recall == 0.0 else 2.0 * precision * recall / (precision + recall)
        recalls.append(float(recall))
        f1_scores.append(float(f1))

    return {
        "logits": logits,
        "labels": true_labels,
        "predictions": predictions,
        "confusion_matrix": confusion,
        "per_class_recall": recalls,
        "accuracy": float((predictions == true_labels).float().mean().item()),
        "macro_f1": float(sum(f1_scores) / num_classes),
    }


# Smoke check
smoke_metrics = evaluate_model(smoke_model, val_loader, NUM_CLASSES)
print("accuracy:", smoke_metrics["accuracy"])
print("Macro-F1:", smoke_metrics["macro_f1"])
print("confusion matrix:\n", smoke_metrics["confusion_matrix"])

## Exercise 25-E: Assemble the full recode experiment

Connect every earlier component into one reproducible pipeline. Re-seed before model construction, keep a loss history, and return the trained artifacts needed for review.

**Return structure — `run_recode_experiment(images, labels, epochs, batch_size, seed)`:**

- Returns a `dict` with exactly four keys.
- `"model"`: the trained `MinimalCNN` instance on CPU.
- `"split"`: the six-key dictionary returned by `split_and_normalize`.
- `"history"`: `list[float]` of length `epochs`, containing finite non-negative mean training losses.
- `"metrics"`: the seven-key dictionary returned by `evaluate_model` for the validation partition.

In [ ]:
def run_recode_experiment(images, labels, epochs=4, batch_size=18, seed=25):
    if epochs < 1 or batch_size < 1:
        raise ValueError("epochs and batch_size must be positive")
    torch.manual_seed(seed)
    split = split_and_normalize(images, labels, val_fraction=0.25, seed=seed)
    training_data = TensorDataset(split["train_images"], split["train_labels"])
    validation_data = TensorDataset(split["val_images"], split["val_labels"])
    training_loader = DataLoader(
        training_data,
        batch_size=batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    validation_loader = DataLoader(validation_data, batch_size=batch_size, shuffle=False)

    model = MinimalCNN(int(labels.max().item()) + 1)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
    criterion = nn.CrossEntropyLoss()
    history = []
    for _ in range(epochs):
        history.append(train_one_epoch(model, training_loader, optimizer, criterion))

    metrics = evaluate_model(model, validation_loader, int(labels.max().item()) + 1)
    return {"model": model, "split": split, "history": history, "metrics": metrics}


# Smoke check
smoke_run = run_recode_experiment(images, labels, epochs=4, batch_size=18, seed=SEED)
print("loss history:", smoke_run["history"])
print("validation Macro-F1:", smoke_run["metrics"]["macro_f1"])

## Test Cases

Run this cell after completing all TODO cells. A correct implementation should print `Day 25 tests passed`.

**Return structure — `run_day25_tests()`:**

- Returns `None`.
- Success is communicated by completing all assertions and printing exactly `Day 25 tests passed`.

In [ ]:
def run_day25_tests():
    assert "split_and_normalize" in globals(), "Missing function: split_and_normalize"
    assert "MinimalCNN" in globals(), "Missing class: MinimalCNN"
    assert "train_one_epoch" in globals(), "Missing function: train_one_epoch"
    assert "evaluate_model" in globals(), "Missing function: evaluate_model"
    assert "run_recode_experiment" in globals(), "Missing function: run_recode_experiment"

    split = split_and_normalize(images, labels, val_fraction=0.25, seed=SEED)
    assert set(split) == {"train_images", "train_labels", "val_images", "val_labels", "mean", "std"}
    assert split["train_images"].ndim == 4 and split["val_images"].ndim == 4
    assert split["train_images"].dtype == torch.float32
    assert split["train_labels"].dtype == torch.long
    assert split["train_images"].device.type == "cpu"
    assert len(split["train_labels"]) + len(split["val_labels"]) == len(labels)
    assert set(split["train_labels"].tolist()) == set(range(NUM_CLASSES))
    assert set(split["val_labels"].tolist()) == set(range(NUM_CLASSES))
    assert abs(split["train_images"].mean().item()) < 1e-5
    assert abs(split["train_images"].std().item() - 1.0) < 1e-4
    assert split["std"].item() > 0.0

    model = MinimalCNN(NUM_CLASSES)
    logits = model(split["train_images"][:5])
    assert logits.shape == (5, NUM_CLASSES)
    assert logits.dtype == torch.float32 and logits.device.type == "cpu"

    loader = DataLoader(TensorDataset(split["train_images"], split["train_labels"]), batch_size=16)
    before = [parameter.detach().clone() for parameter in model.parameters()]
    loss = train_one_epoch(model, loader, torch.optim.Adam(model.parameters(), lr=0.02), nn.CrossEntropyLoss())
    assert isinstance(loss, float) and np.isfinite(loss) and loss >= 0.0
    assert any(not torch.equal(old, new.detach()) for old, new in zip(before, model.parameters()))

    val = DataLoader(TensorDataset(split["val_images"], split["val_labels"]), batch_size=16)
    metrics = evaluate_model(model, val, NUM_CLASSES)
    assert set(metrics) == {"logits", "labels", "predictions", "confusion_matrix", "per_class_recall", "accuracy", "macro_f1"}
    assert metrics["logits"].shape == (len(split["val_labels"]), NUM_CLASSES)
    assert metrics["labels"].dtype == torch.long and metrics["predictions"].dtype == torch.long
    assert metrics["confusion_matrix"].shape == (NUM_CLASSES, NUM_CLASSES)
    assert metrics["confusion_matrix"].dtype == torch.long
    assert int(metrics["confusion_matrix"].sum()) == len(split["val_labels"])
    assert len(metrics["per_class_recall"]) == NUM_CLASSES
    assert 0.0 <= metrics["accuracy"] <= 1.0
    assert 0.0 <= metrics["macro_f1"] <= 1.0

    result = run_recode_experiment(images, labels, epochs=4, batch_size=18, seed=SEED)
    assert set(result) == {"model", "split", "history", "metrics"}
    assert isinstance(result["model"], MinimalCNN)
    assert len(result["history"]) == 4
    assert all(isinstance(value, float) and np.isfinite(value) and value >= 0.0 for value in result["history"])
    assert result["history"][-1] < result["history"][0]
    assert result["metrics"]["macro_f1"] >= 0.90

    print("Day 25 tests passed")


run_day25_tests()

## Day 25 Checklist

- [ ] I can state every important tensor shape, dtype, and device.
- [ ] I split data before calculating preprocessing statistics.
- [ ] I can rebuild a small CNN without copying an old implementation.
- [ ] I switch correctly between training and evaluation modes.
- [ ] I calculate accuracy, confusion matrix, per-class recall, and Macro-F1.
- [ ] I can rerun the experiment with the same seed and explain the result.
- [ ] I recorded the parts I could not recode from memory for targeted review.